In [21]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')

In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [23]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [24]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [25]:
multiply.name

'multiply'

In [26]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [27]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [28]:
# tool binding

In [29]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', api_key=api_key)

In [31]:
llm.invoke('hi')

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c4a98-4e46-7160-bd12-5d5d65b9ac18-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 33, 'total_tokens': 35, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 23}})

In [32]:
llm_with_tools = llm.bind_tools([multiply])

In [33]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing well, thank you! How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c4a98-6240-7f51-9d93-be045163377c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 16, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})

In [34]:
query = HumanMessage('can you multiply 3 with 1000')

In [35]:
messages = [query]

In [36]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [37]:
result = llm_with_tools.invoke(messages)

In [38]:
messages.append(result)

In [39]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 1000}'}, '__gemini_function_call_thought_signatures__': {'6d47b7b4-7a14-43fd-8153-06f481f9ac86': 'CpUCAb4+9vtLBXgbAnszq5ohHnCoxiHtN+sCKKCeEvqgdFGZyEDNHmANUdTKpwJV+CeSRuB1wxpPi5uzeTN72IUv+quXOX94pHn27sGn19hT37Mppd5SZ/ZT5CFeb47hfAv0WxAQCHlZafM058m+tF1txrDSkb/BhvrXbfX7IKN4tLSZQfY+esmc68JVH9TTv9LP8gsh9jO4xq/lzW9bjQLxL4Rma1Bht/j/ku/OyOl6uDMAChnrgqPFcmeAEs018Y5AbnDlJf+7Es8PCRvmXM9bVEULRg7v+NA7Xwbmgxex40jyfwfgbzJtgdjZpK+nta5dkqTy1eYOIhD5UkbINvcNvfH/3W1vsHiRDZydiwQXU10jK12zZw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c4a98-7162-77e2-8bbe-f428005ae9fe-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': '6d47b7b4-7a14-43fd-8153-06f481f9ac86', 'type': 'tool_call

In [43]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 1000},
  'id': '6d47b7b4-7a14-43fd-8153-06f481f9ac86',
  'type': 'tool_call'}]

In [42]:
# we have to sent the args only right
multiply.invoke(result.tool_calls[0]['args'])

3000

In [45]:
# but if we send the entire tool call msg
multiply.invoke(result.tool_calls[0])

ToolMessage(content='3000', name='multiply', tool_call_id='6d47b7b4-7a14-43fd-8153-06f481f9ac86')

In [46]:
tool_result = multiply.invoke(result.tool_calls[0])

In [47]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='6d47b7b4-7a14-43fd-8153-06f481f9ac86')

In [48]:
messages.append(tool_result)

In [49]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 1000}'}, '__gemini_function_call_thought_signatures__': {'6d47b7b4-7a14-43fd-8153-06f481f9ac86': 'CpUCAb4+9vtLBXgbAnszq5ohHnCoxiHtN+sCKKCeEvqgdFGZyEDNHmANUdTKpwJV+CeSRuB1wxpPi5uzeTN72IUv+quXOX94pHn27sGn19hT37Mppd5SZ/ZT5CFeb47hfAv0WxAQCHlZafM058m+tF1txrDSkb/BhvrXbfX7IKN4tLSZQfY+esmc68JVH9TTv9LP8gsh9jO4xq/lzW9bjQLxL4Rma1Bht/j/ku/OyOl6uDMAChnrgqPFcmeAEs018Y5AbnDlJf+7Es8PCRvmXM9bVEULRg7v+NA7Xwbmgxex40jyfwfgbzJtgdjZpK+nta5dkqTy1eYOIhD5UkbINvcNvfH/3W1vsHiRDZydiwQXU10jK12zZw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c4a98-7162-77e2-8bbe-f428005ae9fe-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': '6d47b7b4-7a14-43fd-8153-06f481f9ac86', 'type': 'tool_call

In [51]:
llm_with_tools.invoke(messages)

AIMessage(content='The product of 3 and 1000 is 3000.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c4aab-c7a7-7363-8a2d-b0fbea8ca580-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 102, 'output_tokens': 18, 'total_tokens': 120, 'input_token_details': {'cache_read': 0}})

## Currency Converter

In [ ]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [ ]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [ ]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1745798401,
 'time_last_update_utc': 'Mon, 28 Apr 2025 00:00:01 +0000',
 'time_next_update_unix': 1745884801,
 'time_next_update_utc': 'Tue, 29 Apr 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 85.4741}

In [ ]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [ ]:
# tool binding
llm = ChatOpenAI()

In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [ ]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [ ]:
ai_message = llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_message)

In [ ]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'call_PKL8v7zwmphzNel0MnvjjGvY',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_vRdld30yHTKFGcTQPumgzH5u',
  'type': 'tool_call'}]

In [ ]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

  

In [ ]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_PKL8v7zwmphzNel0MnvjjGvY', 'function': {'arguments': '{"base_currency": "INR", "target_currency": "USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_vRdld30yHTKFGcTQPumgzH5u', 'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 123, 'total_tokens': 176, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BR8rovupYuNnIbGO41TOO3knQ2Cr9', 'finish_reason': 'too

In [ ]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.0117. \n\nBased on this conversion factor, 10 INR is equal to 0.117 USD.'

In [ ]:
from langchain.agents import initialize_agent, AgentType

# Step 5: Initialize the Agent ---
agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,  # using ReAct pattern
    verbose=True  # shows internal thinking
)

In [ ]:
# --- Step 6: Run the Agent ---
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})



> Entering new AgentExecutor chain...
I'm here and ready to help! How can I assist you today?

> Finished chain.
